In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import yaml

PROJECT_ROOT = Path.home() / "mlops"

print("Project:", PROJECT_ROOT)
print("MLflow:", mlflow.__version__)

Project: /home/student/mlops
MLflow: 3.17.1


In [2]:
run_dir = PROJECT_ROOT / "runs/lab03_script/yolo11n_construction_ppe_small"

results_csv = run_dir / "results.csv"

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

print("Run directory:", run_dir)
print("Epochs:", len(df))

display(df.head())

Run directory: /home/student/mlops/runs/lab03_script/yolo11n_construction_ppe_small
Epochs: 10


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
0,1,72.0116,1.72698,3.92421,1.35428,0.17395,0.07481,0.08076,0.05531,1.56014,3.16402,1.24206,0.000206,0.000206,0.000206
1,2,142.4190,1.61009,3.13125,1.25219,0.27997,0.11310,0.12799,0.08407,1.58255,2.65012,1.22311,0.000374,0.000374,0.000374
2,3,210.8450,1.63584,2.62903,1.19840,0.36773,0.16161,0.17394,0.11492,1.52066,2.29897,1.20614,0.000500,0.000500,0.000500
3,4,277.5630,1.61461,2.48278,1.17838,0.39437,0.21233,0.19130,0.12707,1.43112,2.06587,1.17450,0.000439,0.000439,0.000439
4,5,346.3020,1.56733,2.19770,1.14429,0.28516,0.24822,0.22928,0.13994,1.37069,1.84270,1.13707,0.000378,0.000378,0.000378


In [3]:
print("Training artifacts:")

for item in sorted(run_dir.iterdir()):
    print(item.name)

print("\nModel weights:")

for item in sorted((run_dir / "weights").iterdir()):
    print(item.name)

Training artifacts:
BoxF1_curve.png
BoxPR_curve.png
BoxP_curve.png
BoxR_curve.png
args.yaml
confusion_matrix.png
confusion_matrix_normalized.png
labels.jpg
results.csv
results.png
run_metadata.json
train_batch0.jpg
train_batch1.jpg
train_batch2.jpg
val_batch0_labels.jpg
val_batch0_pred.jpg
val_batch1_labels.jpg
val_batch1_pred.jpg
val_batch2_labels.jpg
val_batch2_pred.jpg
weights

Model weights:
best.pt
last.pt


In [4]:
MLFLOW_DIR = PROJECT_ROOT / "data/mlflow-local"
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = MLFLOW_DIR / "mlflow.db"
ARTIFACT_PATH = MLFLOW_DIR / "artifacts"
ARTIFACT_PATH.mkdir(exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:///{DB_PATH.as_posix()}")

EXPERIMENT = "Construction-PPE-YOLO-Notebook"

if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(
        name=EXPERIMENT,
        artifact_location=ARTIFACT_PATH.as_uri()
    )

mlflow.set_experiment(EXPERIMENT)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT)

2026/11/18 15:56:25 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/18 15:56:26 INFO mlflow.store.db.utils: Updating database tables


Tracking URI: sqlite:////home/student/mlops/data/mlflow-local/mlflow.db
Experiment: Construction-PPE-YOLO-Notebook


In [5]:
import subprocess

def git_info(*args):
    return subprocess.check_output(
        ["git", *args],
        cwd=PROJECT_ROOT,
        text=True
    ).strip()

GIT_COMMIT = git_info("rev-parse", "HEAD")
GIT_BRANCH = git_info("branch", "--show-current")
GIT_DIRTY = bool(git_info("status", "--porcelain"))

dvc_file = PROJECT_ROOT / "datasets/construction-ppe-small.dvc"

dataset_revision = yaml.safe_load(
    dvc_file.read_text()
)["outs"][0]["md5"]

print("Git commit:", GIT_COMMIT)
print("Git branch:", GIT_BRANCH)
print("Git dirty:", GIT_DIRTY)
print("Dataset revision:", dataset_revision)

Git commit: 6fb04d346843d9fa7afe30c7756a681872e78d4a
Git branch: main
Git dirty: True
Dataset revision: 170216f9c0aba33c4bda391e8c32f999.dir


In [6]:
args = yaml.safe_load(
    (run_dir / "args.yaml").read_text()
)

with mlflow.start_run(run_name="yolo11n-notebook-01") as run:
    mlflow.log_params({
        "model": args["model"],
        "epochs": args["epochs"],
        "imgsz": args["imgsz"],
        "batch": args["batch"],
        "device": args["device"]
    })

    mlflow.set_tags({
        "dataset.name": "Construction-PPE-small",
        "dataset.revision": dataset_revision,
        "git.commit": GIT_COMMIT,
        "git.branch": GIT_BRANCH,
        "git.dirty": str(GIT_DIRTY).lower(),
        "model.family": "YOLO11",
        "task": "object_detection",
        "environment": "jupyter",
        "source": "lab03-script"
    })

    MLFLOW_RUN_ID = run.info.run_id

print("MLflow Run ID:", MLFLOW_RUN_ID)
print("Parameters and tags saved.")

MLflow Run ID: 64e8e3251bd8494d83e1c437afa61051
Parameters and tags saved.


In [7]:
import math

metric_columns = {
    "precision": "metrics/precision(B)",
    "recall": "metrics/recall(B)",
    "mAP50": "metrics/mAP50(B)",
    "mAP50_95": "metrics/mAP50-95(B)"
}

with mlflow.start_run(run_id=MLFLOW_RUN_ID):
    for index, row in df.iterrows():
        epoch = int(row["epoch"])

        for name, column in metric_columns.items():
            value = float(row[column])

            if math.isfinite(value):
                mlflow.log_metric(name, value, step=epoch)

print("Metrics logged successfully.")
print("Epochs:", len(df))
print("Metrics:", list(metric_columns))

Metrics logged successfully.
Epochs: 10
Metrics: ['precision', 'recall', 'mAP50', 'mAP50_95']


In [8]:
files = {
    "results.csv": "metrics",
    "results.png": "plots",
    "confusion_matrix.png": "plots",
    "confusion_matrix_normalized.png": "plots",
    "BoxF1_curve.png": "plots",
    "BoxPR_curve.png": "plots",
    "BoxP_curve.png": "plots",
    "BoxR_curve.png": "plots",
    "args.yaml": "metadata",
    "run_metadata.json": "metadata",
    "weights/best.pt": "model",
    "weights/last.pt": "model"
}

with mlflow.start_run(run_id=MLFLOW_RUN_ID):
    for filename, destination in files.items():
        path = run_dir / filename

        if path.is_file():
            mlflow.log_artifact(
                str(path),
                artifact_path=destination
            )
            print("Saved:", filename)

print("Artifact logging completed.")

Saved: results.csv
Saved: results.png
Saved: confusion_matrix.png
Saved: confusion_matrix_normalized.png
Saved: BoxF1_curve.png
Saved: BoxPR_curve.png
Saved: BoxP_curve.png
Saved: BoxR_curve.png
Saved: args.yaml
Saved: run_metadata.json
Saved: weights/best.pt
Saved: weights/last.pt
Artifact logging completed.


In [9]:
from ultralytics import YOLO, settings

settings.update({"mlflow": False})

model_second = YOLO(str(PROJECT_ROOT / "yolo11n.pt"))

model_second.train(
    data=str(PROJECT_ROOT / "datasets/construction-ppe-small/data.yaml"),
    epochs=10,
    imgsz=320,
    batch=1,
    device="cpu",
    workers=0,
    seed=args.get("seed", 0),
    project=str(PROJECT_ROOT / "runs/lab04_notebook"),
    name="yolo11n_imgsz320"
)

run_dir_second = Path(model_second.trainer.save_dir)

print("Results:", run_dir_second)

Ultralytics 8.4.215 🚀 Python-3.14.4 torch-2.15.0+cpu CPU (11th Gen Intel Core i7-11370H @ 3.30GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/student/mlops/datasets/construction-ppe-small/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/student/mlops/yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=

In [10]:
results_csv_second = run_dir_second / "results.csv"

df_second = pd.read_csv(results_csv_second)
df_second.columns = df_second.columns.str.strip()

args_second = yaml.safe_load(
    (run_dir_second / "args.yaml").read_text()
)

print("Run directory:", run_dir_second)
print("Epochs:", len(df_second))

display(df_second)

Run directory: /home/student/mlops/runs/lab04_notebook/yolo11n_imgsz320
Epochs: 10


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
0,1,89.3079,1.91683,3.74516,1.39651,0.24781,0.05994,0.07085,0.04517,1.74253,3.14025,1.26679,0.000206,0.000206,0.000206
1,2,160.6110,1.83387,3.16138,1.26261,0.08200,0.09868,0.07620,0.04330,1.79618,2.85395,1.26057,0.000374,0.000374,0.000374
2,3,210.0000,1.75627,2.70482,1.21470,0.26199,0.15686,0.10854,0.05951,1.74246,2.51235,1.23421,0.000500,0.000500,0.000500
3,4,258.9580,1.75261,2.44638,1.19065,0.51633,0.15745,0.15988,0.09208,1.67066,2.22755,1.18972,0.000439,0.000439,0.000439
4,5,307.5360,1.69560,2.17914,1.14272,0.30022,0.23182,0.19709,0.12141,1.56706,1.85793,1.13047,0.000378,0.000378,0.000378
5,6,357.6890,1.67420,2.08073,1.16071,0.31075,0.25555,0.20029,0.12107,1.58883,1.82347,1.13692,0.000316,0.000316,0.000316
6,7,407.3840,1.60044,1.89128,1.12585,0.38715,0.23980,0.22009,0.12677,1.59673,1.76461,1.15109,0.000254,0.000254,0.000254
7,8,455.2120,1.60586,1.80945,1.13043,0.34977,0.26826,0.23436,0.13037,1.57518,1.71684,1.14389,0.000192,0.000192,0.000192
8,9,502.6760,1.54797,1.85534,1.10985,0.38504,0.25962,0.24036,0.13696,1.50355,1.63780,1.10966,0.000130,0.000130,0.000130
9,10,550.3760,1.56677,1.83708,1.10059,0.33960,0.28929,0.24502,0.14009,1.49038,1.63099,1.10745,0.000068,0.000068,0.000068


In [11]:
mlflow.set_experiment(EXPERIMENT)

with mlflow.start_run(run_name="yolo11n-notebook-02") as run:

    mlflow.log_params({
        "model": args_second["model"],
        "epochs": args_second["epochs"],
        "imgsz": args_second["imgsz"],
        "batch": args_second["batch"],
        "device": args_second["device"]
    })

    mlflow.set_tags({
        "dataset.name": "Construction-PPE-small",
        "dataset.revision": dataset_revision,
        "git.commit": GIT_COMMIT,
        "git.branch": GIT_BRANCH,
        "git.dirty": str(GIT_DIRTY).lower(),
        "model.family": "YOLO11",
        "task": "object_detection",
        "environment": "jupyter",
        "source": "lab04-notebook"
    })

    for index, row in df_second.iterrows():
        epoch = int(row["epoch"])

        for name, column in metric_columns.items():
            value = float(row[column])

            if math.isfinite(value):
                mlflow.log_metric(name, value, step=epoch)

    MLFLOW_RUN_ID_SECOND = run.info.run_id

print("MLflow Run ID:", MLFLOW_RUN_ID_SECOND)
print("Parameters, tags and metrics saved.")

MLflow Run ID: 3fbcea0d80f543a8ab08e75ef0ceca58
Parameters, tags and metrics saved.


In [12]:
with mlflow.start_run(run_id=MLFLOW_RUN_ID_SECOND):

    for filename, destination in files.items():
        path = run_dir_second / filename

        if path.is_file():
            mlflow.log_artifact(
                str(path),
                artifact_path=destination
            )
            print("Saved:", filename)

print("Artifact logging completed.")

Saved: results.csv
Saved: results.png
Saved: confusion_matrix.png
Saved: confusion_matrix_normalized.png
Saved: BoxF1_curve.png
Saved: BoxPR_curve.png
Saved: BoxP_curve.png
Saved: BoxR_curve.png
Saved: args.yaml
Saved: weights/best.pt
Saved: weights/last.pt
Artifact logging completed.
